# Reading Raw Sequence Data

Every tool in this course — QIIME2, DADA2, MetaPhlAn3, GMWI2 — ultimately
starts from the same raw material: short strings of A/C/G/T that come out
of a DNA sequencer. Before touching any pipeline, it's worth handling that
raw material yourself, with real code, using **Biopython** — one of the
few genuine bioinformatics libraries that runs directly in this browser
(no install cell needed, it's bundled).

## 1. A sequence is just a string, with biology attached

In [ ]:
from Bio.Seq import Seq

# a short stand-in for one 16S rRNA gene fragment (real reads are ~150-300 bases;
# this is trimmed down so the whole notebook stays readable)
read = Seq("AGAGTTTGATCCTGGCTCAGGACGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGAACGGCAGCACG")
print("length:", len(read))
print(read)

`Seq` objects behave like strings but understand biology. Two operations
you'll use constantly:

- **Reverse complement** — sequencers read both strands, and the strand
  you get isn't always the "forward" one you want to compare against a
  reference.
- **GC content** — the percentage of bases that are G or C. It varies by
  organism and is one of the oldest, simplest sanity checks in the field
  (extreme GC content on a "human" read is a sign something's wrong).

In [ ]:
print("reverse complement:", read.reverse_complement())

gc_count = read.count("G") + read.count("C")
gc_content = 100 * gc_count / len(read)
print(f"GC content: {gc_content:.1f}%")

### 🔧 YOUR TURN #1
Reverse-complement the reverse complement (`read.reverse_complement().reverse_complement()`).
**Predict first:** what should you get back?

In [ ]:
# Your code here

## 2. From DNA to protein: reading frames

Not every pipeline in this course cares about protein — MetaPhlAn3 and
GMWI2 work from marker genes and taxonomy, not translation. But
understanding that a DNA sequence *can* be read as protein, three bases
(a **codon**) at a time, is worth doing once, since it comes up constantly
in the wider field (e.g. functional-prediction tools like PICRUSt2,
mentioned in notebook 07).

In [ ]:
protein = read.translate()
print(protein)

Notice the `*` characters — those are STOP codons. Real coding sequences
are trimmed to start at a start codon and end at a stop codon; a random
raw read like this one isn't a clean coding sequence, so translating it
directly produces nonsense stops mid-sequence. That's expected here — the
point is mechanics, not a real protein.

## 3. A FASTA file is just many sequences, with names

Real datasets don't hand you one sequence — they hand you thousands, in a
**FASTA** file: a `>` header line, then the sequence, repeated. Let's
build a tiny one in memory and parse it the same way you'd parse a file
with `Bio.SeqIO`.

In [ ]:
import io
from Bio import SeqIO

fasta_text = """>sample1_read1
AGAGTTTGATCCTGGCTCAGGACGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGAACGGCAGCACG
>sample1_read2
TTGGGCGTAAAGCGCGCGTAGGCGGCTTTTTAAGTCTGATGTGAAAGCCCACGGCTCAACCGTGGAGGG
>sample1_read3
GGCTGCGGCGCATTAGCTAGTTGGTGGGGTAACGGCTCACCAAGGCGACGATCAGTAGCTGGTCTGAGA
"""

records = list(SeqIO.parse(io.StringIO(fasta_text), "fasta"))
for rec in records:
    gc = 100 * (rec.seq.count("G") + rec.seq.count("C")) / len(rec.seq)
    print(f"{rec.id}: {len(rec.seq)} bp, GC={gc:.1f}%")

### EXPLAIN #1
*Every notebook after this one treats sequences as already resolved into
species names and abundance numbers — you never see a raw base again.
Given what you just did by hand (parsing, orienting, checking GC content),
what do you think a real pipeline does with millions of reads like these
before it ever produces a taxonomy table?*

> your answer here

### 🔧 YOUR TURN #2
Add a fourth record to `fasta_text` (invent any ~60-70 base sequence),
re-run the parse cell, and confirm it shows up in the loop's output.

In [ ]:
# Your code here

## Done — you've touched the raw material

Every abundance table, every taxonomy assignment, every GMWI2 score in
this course started as sequences exactly like these — just billions more
of them, run through software instead of a notebook cell.

**Next:** `02_from_reads_to_taxonomy_table.ipynb` — how millions of reads
like these actually become a table of species names.